# Temporal Retweet Graph — EDA
Covers: tweet volume over time, temporal train/test split, first-retweet-time distribution, degree distributions in history vs future, and LP task feasibility.

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch

CSV_GLOB   = "/project2/ll_774_951/midterm/*/*.csv"
GRAPH_PATH = "/scratch1/eibl/data/midterm/graphs/retweet_graph.pt"
HISTORY_FRACTION = 0.80

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Load raw CSVs

In [ ]:
files = sorted(glob.glob(CSV_GLOB))
print(f"Found {len(files)} CSV files")

chunks = []
for f in files:
    chunks.append(pd.read_csv(f, usecols=['userid', 'rt_userid', 'date', 'tweet_type']))
df = pd.concat(chunks, ignore_index=True)
print(f"Total rows: {len(df):,}")

df['timestamp'] = pd.to_datetime(
    df['date'].astype(str).str.strip(),
    format='%a %b %d %H:%M:%S +0000 %Y',
    utc=True,
    errors='coerce',
)
n_bad = df['timestamp'].isna().sum()
print(f"Rows with invalid timestamp (dropped): {n_bad:,}")
df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
print(f"Date range: {df['timestamp'].min()}  →  {df['timestamp'].max()}")

## 2. Tweet volume over time (histogram)
Shows when the data was collected and where the 80/20 temporal split falls.

In [ ]:
cutoff_idx  = int(len(df) * HISTORY_FRACTION)
cutoff_time = df['timestamp'].iloc[cutoff_idx]
print(f"80% cutoff: row {cutoff_idx:,}  time={cutoff_time}")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, freq, title in zip(axes, ['1h', '10min'], ['Hourly tweet volume', '10-min tweet volume']):
    hist = df.set_index('timestamp').resample(freq).size()
    ax.bar(hist.index.to_pydatetime(), hist.values, width=pd.Timedelta(freq).total_seconds() / 86400,
           align='edge', color='steelblue', alpha=0.8)
    ax.axvline(cutoff_time.to_pydatetime(), color='crimson', lw=2, label=f'80% cutoff ({cutoff_time.strftime("%b %d %H:%M UTC")})')
    ax.set_ylabel('Tweets')
    ax.set_title(title)
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:%M'))

plt.suptitle('Tweet volume over time — red line = train/future split', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nHistory rows : {cutoff_idx:,}")
print(f"Future rows  : {len(df) - cutoff_idx:,}")

## 3. Tweet type breakdown over time

In [ ]:
type_counts = df['tweet_type'].value_counts()
print("Tweet types:")
print(type_counts.to_string())

rt_df = df[df['rt_userid'].notna()].copy()
print(f"\nRetweet rows: {len(rt_df):,} ({100*len(rt_df)/len(df):.1f}% of total)")
print(f"Unique retweeting users  : {rt_df['userid'].nunique():,}")
print(f"Unique retweeted accounts: {rt_df['rt_userid'].nunique():,}")

## 4. First-retweet-time distribution between user pairs
For each directed pair (retweeter → retweeted), how long after the first retweet in history does a retweet show up in the future window?

In [ ]:
rt_df['userid']    = pd.to_numeric(rt_df['userid'],    errors='coerce')
rt_df['rt_userid'] = pd.to_numeric(rt_df['rt_userid'], errors='coerce')
rt_df = rt_df.dropna(subset=['userid', 'rt_userid']).copy()
rt_df['userid']    = rt_df['userid'].astype(np.int64)
rt_df['rt_userid'] = rt_df['rt_userid'].astype(np.int64)

history_rt = rt_df[rt_df['timestamp'] < cutoff_time].copy()
future_rt  = rt_df[rt_df['timestamp'] >= cutoff_time].copy()

# First occurrence of each directed pair in each window
hist_first = history_rt.groupby(['userid', 'rt_userid'])['timestamp'].min().rename('hist_first')
fut_first  = future_rt.groupby(['userid', 'rt_userid'])['timestamp'].min().rename('fut_first')

paired = pd.concat([hist_first, fut_first], axis=1).dropna()
paired['gap_hours'] = (paired['fut_first'] - paired['hist_first']).dt.total_seconds() / 3600

print(f"Pairs with retweets in BOTH windows: {len(paired):,}")
print(f"Pairs in history only              : {len(hist_first) - len(paired):,}")
print(f"Pairs in future only (new links)   : {len(fut_first) - len(paired):,}")
print(f"\nGap (hours) stats:")
print(paired['gap_hours'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(paired['gap_hours'].clip(upper=paired['gap_hours'].quantile(0.99)), bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Hours between first history retweet and first future retweet')
axes[0].set_ylabel('Pairs')
axes[0].set_title('Gap distribution (clipped at 99th pct)')

axes[1].hist(np.log1p(paired['gap_hours']), bins=60, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log(gap_hours + 1)')
axes[1].set_title('Gap distribution (log scale)')

plt.suptitle('Time between first history retweet and first future retweet (same pair)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Degree distribution: history vs future graph
Uses the pre-built `temporal_history` and `temporal_new` edge views from the graph file.

In [ ]:
raw = torch.load(GRAPH_PATH, map_location='cpu')
print("Graph keys:", [k for k in raw.keys() if not isinstance(raw[k], dict)])
print("Edge index views:", list(raw.get('edge_index_views', {}).keys()))
print("Target edge views:", list(raw.get('target_edge_index_views', {}).keys()))

N = raw['x'].shape[0]
ei_hist = raw['edge_index_views']['temporal_history']  # [2, E_hist]
ei_fut  = raw['target_edge_index_views']['temporal_new']  # [2, E_fut]

print(f"\nNodes           : {N:,}")
print(f"History edges   : {ei_hist.shape[1]:,}  ({ei_hist.shape[1]//2:,} undirected)")
print(f"Future new edges: {ei_fut.shape[1]:,}  ({ei_fut.shape[1]//2:,} undirected)")

In [ ]:
def degree_stats(edge_index, n, label):
    out_deg = torch.bincount(edge_index[0], minlength=n).numpy()
    in_deg  = torch.bincount(edge_index[1], minlength=n).numpy()
    print(f"\n{label}")
    print(f"  Out-degree — mean={out_deg.mean():.2f}  median={np.median(out_deg):.0f}  max={out_deg.max()}  isolated={( out_deg==0).sum():,}")
    print(f"  In-degree  — mean={in_deg.mean():.2f}  median={np.median(in_deg):.0f}  max={in_deg.max()}  isolated={(in_deg==0).sum():,}")
    return out_deg, in_deg

out_h, in_h = degree_stats(ei_hist, N, "HISTORY (temporal_history)")
out_f, in_f = degree_stats(ei_fut,  N, "FUTURE  (temporal_new)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, deg, title in zip(
    axes.flat,
    [out_h, in_h, out_f, in_f],
    ['History out-degree', 'History in-degree', 'Future out-degree', 'Future in-degree'],
):
    nonzero = deg[deg > 0]
    ax.hist(np.log1p(nonzero), bins=60, color='steelblue', edgecolor='white')
    ax.set_title(f'{title}  (non-isolated nodes only, log scale)')
    ax.set_xlabel('log(degree + 1)')
    ax.set_ylabel('Nodes')
    ax.axvline(np.log1p(np.mean(nonzero)), color='crimson', lw=1.5, label=f'mean={np.mean(nonzero):.1f}')
    ax.legend(fontsize=9)

plt.suptitle('Degree distributions: history vs future edge views', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Overlay: history vs future in-degree on same axis (log-log)
fig, ax = plt.subplots(figsize=(8, 5))
for deg, label, color in [(in_h, 'History in-degree', 'steelblue'), (in_f, 'Future in-degree', 'darkorange')]:
    nonzero = np.sort(deg[deg > 0])[::-1]
    ranks = np.arange(1, len(nonzero) + 1)
    ax.loglog(ranks, nonzero, '.', alpha=0.3, ms=2, color=color, label=label)

ax.set_xlabel('Rank')
ax.set_ylabel('In-degree')
ax.set_title('In-degree rank plot (log-log) — history vs future')
ax.legend()
plt.tight_layout()
plt.show()

## 6. LP task feasibility: retweeter count distribution per center
This directly determines how many centers can support clean (leak-free) episodes for a given n_shots + n_query.

In [ ]:
from torch_sparse import SparseTensor
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'prodigy'))
from experiments.sampler import preprocess

whole_adj = preprocess(ei_fut, N, bidirectional=True)
rowptr, col, value = whole_adj.csr()

retweeter_counts = []
for center in range(N):
    start, end = int(rowptr[center]), int(rowptr[center + 1])
    if end <= start:
        continue
    edge_ids = value[start:end]
    neigh    = col[start:end]
    incoming = set(neigh[edge_ids < 0].tolist()) - {center}
    if incoming:
        retweeter_counts.append(len(incoming))

rc = np.array(retweeter_counts)
print(f"Centers with ≥1 future retweeter: {len(rc):,} / {N:,} ({100*len(rc)/N:.1f}%)")
print(f"Retweeter count — mean={rc.mean():.2f}  median={np.median(rc):.0f}  max={rc.max()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(rc.clip(max=int(np.percentile(rc, 95))), bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Future retweeters per center')
axes[0].set_ylabel('Centers')
axes[0].set_title('Retweeter count dist (clipped at 95th pct)')

axes[1].hist(np.log1p(rc), bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log(retweeters + 1)')
axes[1].set_title('Retweeter count dist (log scale)')

plt.suptitle('Future retweeters per center node — determines LP task feasibility', fontsize=12)
plt.tight_layout()
plt.show()

# How many centers are usable for each n_shots + n_query budget?
print("\nEligible centers (no leakage) by num_member = n_shots + n_query:")
print(f"{'num_member':>12} {'eligible':>10} {'pct of usable':>14}")
for m in [1, 2, 3, 5, 6, 10, 15, 20]:
    eligible = (rc >= m).sum()
    print(f"{m:>12} {eligible:>10,} {100*eligible/len(rc):>13.1f}%")

## 7. Edge overlap: how many future edges were already in history?
Measures how much the graph structure recurs — if most future edges are new, the LP task is harder.

In [ ]:
def edge_set(ei):
    return set(zip(ei[0].tolist(), ei[1].tolist()))

hist_edges = edge_set(ei_hist)
fut_edges  = edge_set(ei_fut)

overlap    = hist_edges & fut_edges
fut_new    = fut_edges  - hist_edges

print(f"History edges (directed)   : {len(hist_edges):,}")
print(f"Future edges  (directed)   : {len(fut_edges):,}")
print(f"Overlap (in both)          : {len(overlap):,}  ({100*len(overlap)/len(fut_edges):.1f}% of future)")
print(f"Purely new future edges    : {len(fut_new):,}  ({100*len(fut_new)/len(fut_edges):.1f}% of future)")
print()
print("Note: target_edge_index_views['temporal_new'] should contain only fut_new.")
print(f"  Actual future_edge_index size: {ei_fut.shape[1]:,}")
print(f"  Computed fut_new size        : {len(fut_new):,}")

## 8. Node feature quick-look
Stats on the 395-dim node features (11 stats + 384 text embeddings).

In [ ]:
x = raw['x'].numpy()
feat_names = list(raw.get('feature_names', []))
stats_names = [n for n in feat_names if not n.startswith('emb_')]
emb_names   = [n for n in feat_names if n.startswith('emb_')]
stats_idx   = [i for i, n in enumerate(feat_names) if not n.startswith('emb_')]

print(f"Total features: {x.shape[1]}  (stats={len(stats_names)}, emb={len(emb_names)})")
print(f"x range: [{x.min():.3f}, {x.max():.3f}]")

if stats_idx:
    x_stats = x[:, stats_idx]
    stats_df = pd.DataFrame(x_stats, columns=stats_names).describe().T[['mean', 'std', 'min', 'max']]
    print("\nStats features:")
    print(stats_df.to_string(float_format='{:.3f}'.format))

# Embedding norms
if emb_names:
    emb_idx = [i for i, n in enumerate(feat_names) if n.startswith('emb_')]
    emb_norms = np.linalg.norm(x[:, emb_idx], axis=1)
    print(f"\nEmbedding L2 norms — mean={emb_norms.mean():.3f}  std={emb_norms.std():.3f}  min={emb_norms.min():.3f}  max={emb_norms.max():.3f}")
    zero_emb = (emb_norms < 1e-6).sum()
    print(f"Zero-embedding nodes (possible missing profiles): {zero_emb:,} ({100*zero_emb/N:.1f}%)")